In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1: Imports & Setup
# ══════════════════════════════════════════════════════════════════════════════
import os, sys, glob, gc, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (14, 6), 'font.size': 11})
sns.set_style('whitegrid')

# ── Project root ──────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR       = os.path.join(PROJECT_ROOT, 'data', 'processed')
WEIGHTS_DIR    = os.path.join(PROJECT_ROOT, 'model_weights')
RESULTS_DIR    = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Feature engineering functions ─────────────────────────────────────────────
from src.features.ofi import compute_multi_level_ofi
from src.features.microstructure import compute_all_features
from src.features.labels import (
    make_regression_labels, make_classification_labels, DEFAULT_HORIZONS
)

HORIZONS = DEFAULT_HORIZONS  # [10, 20, 50, 100]

# ── Discover tickers ──────────────────────────────────────────────────────────
tickers = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
    and glob.glob(os.path.join(DATA_DIR, d, '*.parquet'))
])

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Tickers      : {tickers}')
print(f'Horizons     : {HORIZONS}')
print(f'Weights dir  : {WEIGHTS_DIR}')

Project root : /home/illionar/Projects/multi-horizon-ofi
Data dir     : /home/illionar/Projects/multi-horizon-ofi/data/processed
Tickers      : ['AAPL', 'AMZN', 'GOOG', 'MSFT', 'TSLA']
Horizons     : [10, 20, 50, 100]
Weights dir  : /home/illionar/Projects/multi-horizon-ofi/model_weights


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2: Data Loading & Feature Engineering Pipeline
# ══════════════════════════════════════════════════════════════════════════════

def build_features_and_labels_from_parquet(
    parquet_path: str,
    ofi_levels: int = 5,
    horizons: list = None,
    alpha: float = 0.5,
) -> tuple:
    """
    Load one day's parquet file and compute all features + labels.
    
    Returns:
        X          : np.ndarray (N, F) — feature matrix
        y_reg      : np.ndarray (N, H) — regression targets (delta mid)
        y_cls      : np.ndarray (N, H) — classification targets (0/1/2)
        feat_names : list of str        — feature column names
    """
    if horizons is None:
        horizons = DEFAULT_HORIZONS
    
    df = pd.read_parquet(parquet_path)
    
    # ── OFI features (ofi_1..5, ofi_cumul_1..5) ─────────────────────────────
    ofi_df = compute_multi_level_ofi(df, max_level=ofi_levels)
    
    # ── Microstructure features ──────────────────────────────────────────────
    micro_df = compute_all_features(df, max_level=ofi_levels)
    
    # ── Raw LOB prices & sizes (levels 1–5) ─────────────────────────────────
    lob_cols = []
    for lvl in range(1, ofi_levels + 1):
        lob_cols.extend([
            f'ask_price_{lvl}', f'ask_size_{lvl}',
            f'bid_price_{lvl}', f'bid_size_{lvl}',
        ])
    
    # ── Combine features ─────────────────────────────────────────────────────
    features = pd.concat([ofi_df, micro_df, df[lob_cols]], axis=1)
    feat_names = list(features.columns)
    
    # ── Labels ───────────────────────────────────────────────────────────────
    reg_df = make_regression_labels(df, horizons)
    cls_df = make_classification_labels(df, horizons, alpha=alpha)
    
    # ── Trim NaN tail (future labels unavailable for last `max_horizon` rows)
    max_h = max(horizons)
    valid_end = len(df) - max_h
    if valid_end <= 0:
        raise ValueError(f'Day has {len(df)} events but max horizon is {max_h}')
    
    X     = features.values[:valid_end].astype(np.float32)
    y_reg = reg_df.values[:valid_end].astype(np.float32)
    y_cls = cls_df.values[:valid_end].astype(np.float32)
    
    return X, y_reg, y_cls, feat_names


def load_ticker_data(
    ticker: str,
    data_dir: str = DATA_DIR,
    horizons: list = None,
    max_days: int = None,
    subsample: int = 1,
) -> tuple:
    """
    Load all (or max_days) parquet files for a ticker.
    Optional subsampling: keep every `subsample`-th row per day.
    
    Returns:
        X, y_reg, y_cls, feature_names
    """
    if horizons is None:
        horizons = DEFAULT_HORIZONS
    
    ticker_dir = os.path.join(data_dir, ticker)
    parquet_files = sorted(glob.glob(os.path.join(ticker_dir, '*.parquet')))
    
    if max_days is not None:
        parquet_files = parquet_files[:max_days]
    
    if not parquet_files:
        raise FileNotFoundError(f'No parquet files in {ticker_dir}')
    
    all_X, all_yr, all_yc = [], [], []
    feat_names = None
    
    for pf in parquet_files:
        X, yr, yc, fnames = build_features_and_labels_from_parquet(
            pf, horizons=horizons
        )
        if subsample > 1:
            X  = X[::subsample]
            yr = yr[::subsample]
            yc = yc[::subsample]
        all_X.append(X)
        all_yr.append(yr)
        all_yc.append(yc)
        if feat_names is None:
            feat_names = fnames
    
    X     = np.concatenate(all_X, axis=0)
    y_reg = np.concatenate(all_yr, axis=0)
    y_cls = np.concatenate(all_yc, axis=0)
    
    # Drop any remaining NaN rows
    mask = ~(np.isnan(X).any(axis=1) | np.isnan(y_reg).any(axis=1) | np.isnan(y_cls).any(axis=1))
    X, y_reg, y_cls = X[mask], y_reg[mask], y_cls[mask]
    
    return X, y_reg, y_cls, feat_names


def temporal_split(X, y_reg, y_cls, train_frac=0.6, val_frac=0.2):
    """
    Temporal split (no shuffling — preserves time order).
    Returns (train, val, test) tuples of (X, y_reg, y_cls).
    """
    n = len(X)
    t1 = int(n * train_frac)
    t2 = int(n * (train_frac + val_frac))
    
    train = (X[:t1],    y_reg[:t1],    y_cls[:t1])
    val   = (X[t1:t2],  y_reg[t1:t2],  y_cls[t1:t2])
    test  = (X[t2:],    y_reg[t2:],    y_cls[t2:])
    return train, val, test


print('✓ Data pipeline functions defined.')

✓ Data pipeline functions defined.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3: Training & Evaluation Helper Functions
# ══════════════════════════════════════════════════════════════════════════════

def train_rf_classifier(
    X_train, y_cls_train, X_val, y_cls_val,
    horizons=HORIZONS,
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42,
):
    """
    Train one RF Classifier per horizon.
    Returns dict: {horizon: fitted_model}
    """
    models = {}
    for i, h in enumerate(horizons):
        y_tr = y_cls_train[:, i].astype(int)
        y_va = y_cls_val[:, i].astype(int)
        
        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight='balanced',
            n_jobs=n_jobs,
            random_state=random_state,
        )
        
        t0 = time.time()
        clf.fit(X_train, y_tr)
        train_time = time.time() - t0
        
        # Validation score
        val_pred = clf.predict(X_val)
        val_f1 = f1_score(y_va, val_pred, average='macro', zero_division=0)
        val_acc = accuracy_score(y_va, val_pred)
        
        models[h] = clf
        print(f'  h={h:>3d}  |  Val Acc={val_acc:.4f}  F1={val_f1:.4f}  |  '
              f'Train time: {train_time:.1f}s  |  n_estimators={n_estimators}')
    
    return models


def train_rf_regressor(
    X_train, y_reg_train, X_val, y_reg_val,
    horizons=HORIZONS,
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42,
):
    """
    Train one RF Regressor per horizon.
    Each model predicts delta_mid_{h} = mid_{t+h} - mid_t.
    Returns dict: {horizon: fitted_model}
    """
    models = {}
    for i, h in enumerate(horizons):
        y_tr = y_reg_train[:, i]
        y_va = y_reg_val[:, i]
        
        reg = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            n_jobs=n_jobs,
            random_state=random_state,
        )
        
        t0 = time.time()
        reg.fit(X_train, y_tr)
        train_time = time.time() - t0
        
        # Validation score
        val_pred = reg.predict(X_val)
        val_r2  = r2_score(y_va, val_pred)
        val_mae = mean_absolute_error(y_va, val_pred)
        
        models[h] = reg
        print(f'  h={h:>3d}  |  Val R²={val_r2:.4f}  MAE={val_mae:.6f}  |  '
              f'Train time: {train_time:.1f}s  |  n_estimators={n_estimators}')
    
    return models


def evaluate_classifier(models, X_test, y_cls_test, horizons=HORIZONS):
    """Evaluate RF classifiers on test set. Returns metrics dict."""
    metrics = {}
    for i, h in enumerate(horizons):
        y_true = y_cls_test[:, i].astype(int)
        y_pred = models[h].predict(X_test)
        
        metrics[f'h{h}_accuracy']        = accuracy_score(y_true, y_pred)
        metrics[f'h{h}_f1_macro']        = f1_score(y_true, y_pred, average='macro', zero_division=0)
        metrics[f'h{h}_f1_weighted']     = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics[f'h{h}_precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics[f'h{h}_recall_macro']    = recall_score(y_true, y_pred, average='macro', zero_division=0)
    
    return metrics


def evaluate_regressor(models, X_test, y_reg_test, horizons=HORIZONS):
    """Evaluate RF regressors on test set. Returns metrics dict."""
    metrics = {}
    for i, h in enumerate(horizons):
        y_true = y_reg_test[:, i]
        y_pred = models[h].predict(X_test)
        
        metrics[f'h{h}_mse']  = float(mean_squared_error(y_true, y_pred))
        metrics[f'h{h}_rmse'] = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        metrics[f'h{h}_mae']  = float(mean_absolute_error(y_true, y_pred))
        metrics[f'h{h}_r2']   = float(r2_score(y_true, y_pred))
        
        # Information Coefficient (Pearson corr between pred and actual)
        if np.std(y_true) > 1e-12 and np.std(y_pred) > 1e-12:
            metrics[f'h{h}_ic'] = float(np.corrcoef(y_true, y_pred)[0, 1])
        else:
            metrics[f'h{h}_ic'] = 0.0
    
    return metrics


def print_clf_metrics(metrics, horizons=HORIZONS):
    """Pretty-print classifier metrics."""
    print(f'{"Horizon":>8s} {"Accuracy":>10s} {"F1-macro":>10s} {"F1-wt":>10s} {"Precision":>10s} {"Recall":>10s}')
    print('─' * 62)
    for h in horizons:
        print(f'  h={h:<4d} '
              f'{metrics[f"h{h}_accuracy"]:10.4f} '
              f'{metrics[f"h{h}_f1_macro"]:10.4f} '
              f'{metrics[f"h{h}_f1_weighted"]:10.4f} '
              f'{metrics[f"h{h}_precision_macro"]:10.4f} '
              f'{metrics[f"h{h}_recall_macro"]:10.4f}')


def print_reg_metrics(metrics, horizons=HORIZONS):
    """Pretty-print regressor metrics."""
    print(f'{"Horizon":>8s} {"MSE":>12s} {"RMSE":>10s} {"MAE":>10s} {"R²":>10s} {"IC":>10s}')
    print('─' * 66)
    for h in horizons:
        print(f'  h={h:<4d} '
              f'{metrics[f"h{h}_mse"]:12.8f} '
              f'{metrics[f"h{h}_rmse"]:10.6f} '
              f'{metrics[f"h{h}_mae"]:10.6f} '
              f'{metrics[f"h{h}_r2"]:10.4f} '
              f'{metrics[f"h{h}_ic"]:10.4f}')


print('✓ Training & evaluation functions defined.')

✓ Training & evaluation functions defined.


---
## ⚡ TRIAL RUN — Error Check (Tiny Data)

Runs the **entire pipeline end-to-end** on ~500 rows to verify:
- Data loading works
- Feature engineering produces correct shapes
- Both RF models train without errors
- Evaluation & saving works

**Run this cell on your laptop first.** If it passes, the full-scale cell below will work on Colab.

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4: ⚡ TRIAL RUN — Validate Pipeline (Tiny Data, ~30 seconds)
# ══════════════════════════════════════════════════════════════════════════════

print('=' * 70)
print('TRIAL RUN — Error checking on tiny data subset')
print('=' * 70)

TRIAL_TICKER = tickers[0]  # First available ticker

# ── 1. Load just 1 day, heavily subsampled ────────────────────────────────────
print(f'\n[1/6] Loading 1 day of {TRIAL_TICKER} (subsampled 1/100)...')
X_trial, y_reg_trial, y_cls_trial, feat_names = load_ticker_data(
    TRIAL_TICKER, max_days=1, subsample=100
)
print(f'  X shape     : {X_trial.shape}')
print(f'  y_reg shape : {y_reg_trial.shape}')
print(f'  y_cls shape : {y_cls_trial.shape}')
print(f'  Features    : {len(feat_names)}')
print(f'  Feature list: {feat_names}')

# ── 2. Temporal split ─────────────────────────────────────────────────────────
print(f'\n[2/6] Temporal split (60/20/20)...')
train_t, val_t, test_t = temporal_split(X_trial, y_reg_trial, y_cls_trial)
X_tr, yr_tr, yc_tr = train_t
X_va, yr_va, yc_va = val_t
X_te, yr_te, yc_te = test_t
print(f'  Train: {X_tr.shape[0]:>6,}  Val: {X_va.shape[0]:>6,}  Test: {X_te.shape[0]:>6,}')

# ── 3. Scale features ─────────────────────────────────────────────────────────
print(f'\n[3/6] Scaling features (StandardScaler fit on train)...')
scaler_trial = StandardScaler()
X_tr_s = scaler_trial.fit_transform(X_tr)
X_va_s = scaler_trial.transform(X_va)
X_te_s = scaler_trial.transform(X_te)
print(f'  Scaler fitted. Mean range: [{X_tr_s.mean(axis=0).min():.2f}, {X_tr_s.mean(axis=0).max():.2f}]')

# ── 4. Train RF Classifier (tiny) ─────────────────────────────────────────────
print(f'\n[4/6] Training RF Classifier (n_estimators=5, max_depth=4)...')
clf_models_trial = train_rf_classifier(
    X_tr_s, yc_tr, X_va_s, yc_va,
    horizons=HORIZONS,
    n_estimators=5,     # tiny for speed
    max_depth=4,        # shallow for speed
    min_samples_leaf=2,
)

# ── 5. Train RF Regressor (tiny) ──────────────────────────────────────────────
print(f'\n[5/6] Training RF Regressor (n_estimators=5, max_depth=4)...')
reg_models_trial = train_rf_regressor(
    X_tr_s, yr_tr, X_va_s, yr_va,
    horizons=HORIZONS,
    n_estimators=5,
    max_depth=4,
    min_samples_leaf=2,
)

# ── 6. Evaluate on test set ───────────────────────────────────────────────────
print(f'\n[6/6] Evaluating on test set...')

clf_metrics_trial = evaluate_classifier(clf_models_trial, X_te_s, yc_te)
reg_metrics_trial = evaluate_regressor(reg_models_trial, X_te_s, yr_te)

print(f'\n── RF Classifier (Trial) ──')
print_clf_metrics(clf_metrics_trial)

print(f'\n── RF Regressor (Trial) ──')
print_reg_metrics(reg_metrics_trial)

# ── 7. Test saving ────────────────────────────────────────────────────────────
trial_save_path = os.path.join(WEIGHTS_DIR, '_trial_test_rf.joblib')
joblib.dump({'test': 'ok'}, trial_save_path)
_ = joblib.load(trial_save_path)
os.remove(trial_save_path)
print(f'\n✅ Save/load test passed.')

# ── Cleanup ───────────────────────────────────────────────────────────────────
del X_trial, y_reg_trial, y_cls_trial
del train_t, val_t, test_t
del X_tr, yr_tr, yc_tr, X_va, yr_va, yc_va, X_te, yr_te, yc_te
del X_tr_s, X_va_s, X_te_s
del clf_models_trial, reg_models_trial
gc.collect()

print('\n' + '=' * 70)
print('✅ TRIAL RUN PASSED — No errors. Safe to proceed to full training.')
print('=' * 70)

TRIAL RUN — Error checking on tiny data subset

[1/6] Loading 1 day of AAPL (subsampled 1/100)...
  X shape     : (18641, 38)
  y_reg shape : (18641, 4)
  y_cls shape : (18641, 4)
  Features    : 38
  Feature list: ['ofi_1', 'ofi_cumul_1', 'ofi_2', 'ofi_cumul_2', 'ofi_3', 'ofi_cumul_3', 'ofi_4', 'ofi_cumul_4', 'ofi_5', 'ofi_cumul_5', 'mid_price', 'spread', 'volume_imbalance', 'weighted_mid_price', 'bid_depth', 'ask_depth', 'total_depth', 'depth_imbalance', 'ask_price_1', 'ask_size_1', 'bid_price_1', 'bid_size_1', 'ask_price_2', 'ask_size_2', 'bid_price_2', 'bid_size_2', 'ask_price_3', 'ask_size_3', 'bid_price_3', 'bid_size_3', 'ask_price_4', 'ask_size_4', 'bid_price_4', 'bid_size_4', 'ask_price_5', 'ask_size_5', 'bid_price_5', 'bid_size_5']

[2/6] Temporal split (60/20/20)...
  Train: 11,184  Val:  3,728  Test:  3,729

[3/6] Scaling features (StandardScaler fit on train)...
  Scaler fitted. Mean range: [-0.00, 0.00]

[4/6] Training RF Classifier (n_estimators=5, max_depth=4)...
  h= 10

---
## 🚀 FULL-SCALE TRAINING

Loads **all tickers × all days**, trains RF models with production hyperparameters.

**Run this on Google Colab** (or any machine with ≥16 GB RAM).

Estimated time: 5–15 minutes depending on data size and CPU cores.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5: 🚀 FULL-SCALE — Load All Data
# ══════════════════════════════════════════════════════════════════════════════

print('=' * 70)
print('FULL-SCALE TRAINING — Loading all data')
print('=' * 70)

t0_load = time.time()

all_X, all_yr, all_yc = [], [], []
feat_names = None

for ticker in tickers:
    print(f'\n  Loading {ticker}...')
    X_t, yr_t, yc_t, fnames = load_ticker_data(
        ticker, horizons=HORIZONS, subsample=1  # full data, no subsampling
    )
    print(f'    {ticker}: {X_t.shape[0]:>10,} rows × {X_t.shape[1]} features')
    all_X.append(X_t)
    all_yr.append(yr_t)
    all_yc.append(yc_t)
    if feat_names is None:
        feat_names = fnames
    del X_t, yr_t, yc_t
    gc.collect()

X_all     = np.concatenate(all_X, axis=0)
y_reg_all = np.concatenate(all_yr, axis=0)
y_cls_all = np.concatenate(all_yc, axis=0)
del all_X, all_yr, all_yc; gc.collect()

load_time = time.time() - t0_load

print(f'\n{"─"*70}')
print(f'Total dataset : {X_all.shape[0]:>12,} rows × {X_all.shape[1]} features')
print(f'Memory        : {X_all.nbytes / 1e9:.2f} GB (features) + '
      f'{(y_reg_all.nbytes + y_cls_all.nbytes) / 1e6:.1f} MB (labels)')
print(f'Load time     : {load_time:.1f}s')
print(f'Features ({len(feat_names)}): {feat_names}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6: Temporal Split & Scaling
# ══════════════════════════════════════════════════════════════════════════════

print('Temporal split (60% train / 20% val / 20% test)...')
train_data, val_data, test_data = temporal_split(X_all, y_reg_all, y_cls_all)

X_train, y_reg_train, y_cls_train = train_data
X_val,   y_reg_val,   y_cls_val   = val_data
X_test,  y_reg_test,  y_cls_test  = test_data

# Free the big concatenated arrays
del X_all, y_reg_all, y_cls_all, train_data, val_data, test_data
gc.collect()

print(f'  Train : {X_train.shape[0]:>10,} rows')
print(f'  Val   : {X_val.shape[0]:>10,} rows')
print(f'  Test  : {X_test.shape[0]:>10,} rows')

# ── Scale features (fit on train only) ───────────────────────────────────────
print('\nScaling features (StandardScaler)...')
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f'  Scaler fitted.')

# Label distribution
print('\nClassification label distribution (train):')
for i, h in enumerate(HORIZONS):
    counts = np.bincount(y_cls_train[:, i].astype(int), minlength=3)
    pcts = counts / counts.sum() * 100
    print(f'  h={h:>3d}  →  DOWN={pcts[0]:.1f}%  STAT={pcts[1]:.1f}%  UP={pcts[2]:.1f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: 🌲 Train RF Classifier (Full Scale)
# ══════════════════════════════════════════════════════════════════════════════

RF_CLF_PARAMS = {
    'n_estimators':    200,
    'max_depth':       16,
    'min_samples_leaf': 20,
    'n_jobs':          -1,
    'random_state':    42,
}

print('=' * 70)
print(f'TRAINING: Random Forest CLASSIFIER')
print(f'  Params: {RF_CLF_PARAMS}')
print('=' * 70)

t0_clf = time.time()
clf_models = train_rf_classifier(
    X_train_s, y_cls_train, X_val_s, y_cls_val,
    horizons=HORIZONS,
    **RF_CLF_PARAMS,
)
clf_train_time = time.time() - t0_clf
print(f'\nTotal classifier training time: {clf_train_time:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: 🌲 Train RF Regressor (Full Scale)
# ══════════════════════════════════════════════════════════════════════════════

RF_REG_PARAMS = {
    'n_estimators':    200,
    'max_depth':       16,
    'min_samples_leaf': 20,
    'n_jobs':          -1,
    'random_state':    42,
}

print('=' * 70)
print(f'TRAINING: Random Forest REGRESSOR')
print(f'  Params: {RF_REG_PARAMS}')
print('=' * 70)

t0_reg = time.time()
reg_models = train_rf_regressor(
    X_train_s, y_reg_train, X_val_s, y_reg_val,
    horizons=HORIZONS,
    **RF_REG_PARAMS,
)
reg_train_time = time.time() - t0_reg
print(f'\nTotal regressor training time: {reg_train_time:.1f}s')

---
## 📊 Evaluation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9: Evaluate Both Models on Test Set
# ══════════════════════════════════════════════════════════════════════════════

print('=' * 70)
print('TEST SET EVALUATION')
print('=' * 70)

# ── Classifier ────────────────────────────────────────────────────────────────
clf_metrics = evaluate_classifier(clf_models, X_test_s, y_cls_test)

print('\n┌──────────────────────────────────────────────────────────────────┐')
print('│  RF CLASSIFIER — Test Metrics                                    │')
print('└──────────────────────────────────────────────────────────────────┘')
print_clf_metrics(clf_metrics)

# Per-horizon classification reports
for i, h in enumerate(HORIZONS):
    y_true = y_cls_test[:, i].astype(int)
    y_pred = clf_models[h].predict(X_test_s)
    print(f'\n── Detailed Report: h={h} ──')
    print(classification_report(y_true, y_pred, target_names=['DOWN', 'STAT', 'UP'], zero_division=0))

# ── Regressor ─────────────────────────────────────────────────────────────────
reg_metrics = evaluate_regressor(reg_models, X_test_s, y_reg_test)

print('\n┌──────────────────────────────────────────────────────────────────┐')
print('│  RF REGRESSOR — Test Metrics                                     │')
print('└──────────────────────────────────────────────────────────────────┘')
print_reg_metrics(reg_metrics)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10: Confusion Matrices
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 5))
label_names = ['DOWN', 'STAT', 'UP']

for i, h in enumerate(HORIZONS):
    y_true = y_cls_test[:, i].astype(int)
    y_pred = clf_models[h].predict(X_test_s)
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                ax=axes[i], vmin=0, vmax=100,
                cbar_kws={'label': '%'})
    axes[i].set_title(f'h = {h}', fontweight='bold', fontsize=13)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('RF Classifier — Confusion Matrices (% per true class)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rf_classifier_confusion.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11: Feature Importance — Both Models
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, len(HORIZONS), figsize=(6 * len(HORIZONS), 14))

for i, h in enumerate(HORIZONS):
    # ── Classifier feature importance ─────────────────────────────────────
    imp_clf = clf_models[h].feature_importances_
    sorted_idx = np.argsort(imp_clf)
    top_n = min(20, len(feat_names))  # show top 20
    top_idx = sorted_idx[-top_n:]
    
    axes[0, i].barh(
        [feat_names[j] for j in top_idx],
        imp_clf[top_idx],
        color='steelblue', edgecolor='none'
    )
    axes[0, i].set_title(f'Classifier h={h}', fontweight='bold')
    axes[0, i].set_xlabel('Gini Importance')
    
    # ── Regressor feature importance ──────────────────────────────────────
    imp_reg = reg_models[h].feature_importances_
    sorted_idx = np.argsort(imp_reg)
    top_idx = sorted_idx[-top_n:]
    
    axes[1, i].barh(
        [feat_names[j] for j in top_idx],
        imp_reg[top_idx],
        color='darkorange', edgecolor='none'
    )
    axes[1, i].set_title(f'Regressor h={h}', fontweight='bold')
    axes[1, i].set_xlabel('Gini Importance')

axes[0, 0].set_ylabel('CLASSIFIER\n\nFeature', fontweight='bold', fontsize=12)
axes[1, 0].set_ylabel('REGRESSOR\n\nFeature', fontweight='bold', fontsize=12)

plt.suptitle('Random Forest — Feature Importance (Top 20 per Horizon)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rf_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12: Regressor — Predicted vs Actual Scatter
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 5))

# Sample points for plotting (avoid overplotting)
n_plot = min(5000, X_test_s.shape[0])
plot_idx = np.random.RandomState(42).choice(X_test_s.shape[0], n_plot, replace=False)

for i, h in enumerate(HORIZONS):
    y_true = y_reg_test[plot_idx, i]
    y_pred = reg_models[h].predict(X_test_s[plot_idx])
    
    axes[i].scatter(y_true, y_pred, alpha=0.15, s=5, color='teal')
    
    # Perfect prediction line
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    axes[i].plot(lims, lims, 'r--', alpha=0.8, linewidth=1.5, label='Perfect')
    
    r2 = reg_metrics[f'h{h}_r2']
    ic = reg_metrics[f'h{h}_ic']
    axes[i].set_title(f'h={h}  (R²={r2:.4f}, IC={ic:.4f})', fontweight='bold')
    axes[i].set_xlabel('Actual ΔMid')
    axes[i].set_ylabel('Predicted ΔMid')
    axes[i].legend(loc='upper left')

plt.suptitle('RF Regressor — Predicted vs Actual Mid-Price Change', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rf_regressor_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 Save Models & Metadata

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 13: Save Models, Scaler & Metadata
# ══════════════════════════════════════════════════════════════════════════════

print('Saving models to model_weights/...')

# ── Save classifier models (one per horizon) ─────────────────────────────────
for h in HORIZONS:
    path = os.path.join(WEIGHTS_DIR, f'rf_classifier_h{h}.joblib')
    joblib.dump(clf_models[h], path)
    size_mb = os.path.getsize(path) / 1e6
    print(f'  ✓ {path}  ({size_mb:.1f} MB)')

# ── Save regressor models (one per horizon) ──────────────────────────────────
for h in HORIZONS:
    path = os.path.join(WEIGHTS_DIR, f'rf_regressor_h{h}.joblib')
    joblib.dump(reg_models[h], path)
    size_mb = os.path.getsize(path) / 1e6
    print(f'  ✓ {path}  ({size_mb:.1f} MB)')

# ── Save scaler ──────────────────────────────────────────────────────────────
scaler_path = os.path.join(WEIGHTS_DIR, 'rf_scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f'  ✓ {scaler_path}')

# ── Save metadata ────────────────────────────────────────────────────────────
metadata = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'tickers': tickers,
    'horizons': HORIZONS,
    'feature_names': feat_names,
    'n_features': len(feat_names),
    'data': {
        'n_train': int(X_train.shape[0]),
        'n_val':   int(X_val.shape[0]),
        'n_test':  int(X_test.shape[0]),
        'n_total': int(X_train.shape[0] + X_val.shape[0] + X_test.shape[0]),
    },
    'classifier': {
        'params': RF_CLF_PARAMS,
        'train_time_s': round(clf_train_time, 1),
        'test_metrics': {k: round(v, 6) for k, v in clf_metrics.items()},
    },
    'regressor': {
        'params': RF_REG_PARAMS,
        'train_time_s': round(reg_train_time, 1),
        'test_metrics': {k: round(v, 6) for k, v in reg_metrics.items()},
    },
}

meta_path = os.path.join(WEIGHTS_DIR, 'rf_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f'  ✓ {meta_path}')

# Also save to results/
results_meta_path = os.path.join(RESULTS_DIR, 'rf_results.json')
with open(results_meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f'  ✓ {results_meta_path}')

print('\n✅ All models and metadata saved successfully.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 14: Summary Comparison Table
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 80)
print('FINAL RESULTS SUMMARY')
print('=' * 80)

print('\n┌─────────────────────────────────────────────────────────────────────┐')
print('│  MODEL 1: RF CLASSIFIER  (Label: DOWN=0, STAT=1, UP=2)             │')
print('│  Input: 36 features (OFI + Micro + Raw LOB)                         │')
print('│  Output: Direction class per horizon                                │')
print('└─────────────────────────────────────────────────────────────────────┘')
print_clf_metrics(clf_metrics)

print('\n┌─────────────────────────────────────────────────────────────────────┐')
print('│  MODEL 2: RF REGRESSOR  (Target: ΔMid = mid_{t+h} − mid_t)         │')
print('│  Input: 36 features (OFI + Micro + Raw LOB)                         │')
print('│  Output: Continuous mid-price change per horizon                    │')
print('└─────────────────────────────────────────────────────────────────────┘')
print_reg_metrics(reg_metrics)

# ── OFI importance summary ────────────────────────────────────────────────────
print('\n' + '=' * 80)
print('OFI FEATURE IMPORTANCE ANALYSIS')
print('=' * 80)

ofi_features = [f for f in feat_names if f.startswith('ofi_')]
micro_features = ['mid_price', 'spread', 'volume_imbalance',
                  'weighted_mid_price', 'bid_depth', 'ask_depth',
                  'total_depth', 'depth_imbalance']
lob_features = [f for f in feat_names if f not in ofi_features and f not in micro_features]

ofi_idx   = [feat_names.index(f) for f in ofi_features if f in feat_names]
micro_idx = [feat_names.index(f) for f in micro_features if f in feat_names]
lob_idx   = [feat_names.index(f) for f in lob_features if f in feat_names]

for h in HORIZONS:
    imp = clf_models[h].feature_importances_
    ofi_imp   = imp[ofi_idx].sum()   if ofi_idx   else 0
    micro_imp = imp[micro_idx].sum() if micro_idx else 0
    lob_imp   = imp[lob_idx].sum()   if lob_idx   else 0
    total_imp = ofi_imp + micro_imp + lob_imp
    
    print(f'  h={h:>3d}  |  '
          f'OFI: {ofi_imp/total_imp*100:5.1f}%  '
          f'Micro: {micro_imp/total_imp*100:5.1f}%  '
          f'Raw LOB: {lob_imp/total_imp*100:5.1f}%')

print('\n' + '=' * 80)
print('Files saved:')
print('  model_weights/rf_classifier_h{10,20,50,100}.joblib')
print('  model_weights/rf_regressor_h{10,20,50,100}.joblib')
print('  model_weights/rf_scaler.joblib')
print('  model_weights/rf_metadata.json')
print('  results/rf_results.json')
print('  results/rf_classifier_confusion.png')
print('  results/rf_feature_importance.png')
print('  results/rf_regressor_scatter.png')
print('=' * 80)